# 🚀 Сборка APK - Магический Артефакт

## Инструкция:
1. **СНАЧАЛА** загрузите ZIP проекта (Files → Upload file)
2. **ЗАТЕМ** запустите ячейки по очереди (Shift+Enter)

Ячейка 1: Распаковка проекта  
Ячейка 2: Установка зависимостей  
Ячейка 3: Android SDK  
Ячейка 4: **СБОРКА APK** (10-20 минут)  
Ячейка 5: Скачивание результата


## ☝️ ПЕРЕД НАЧАЛОМ:

1. На своем ПК сожмите папку `opencode` в ZIP
2. В Colab нажмите `Files` (слева) → `Upload file`
3. Выберите `opencode.zip`
4. Дождитесь загрузки
5. **ПОТОМ** запустите первую ячейку ниже

In [ ]:
# ============================================
# ЯЧЕЙКА 1: РАСПАКОВКА ПРОЕКТА
# ============================================

import zipfile
import os
import glob

print("Поиск ZIP файла...")

# Ищем ZIP в /content
zip_files = glob.glob('/content/*.zip')

if not zip_files:
    print("❌ ОШИБКА: ZIP файл не найден!")
    print("")
    print("Действия:")
    print("1. На своем ПК: сожмите папку opencode в ZIP")
    print("2. В Colab: Files (слева) → Upload file")
    print("3. Выберите opencode.zip")
    print("4. Дождитесь загрузки")
    print("5. Запустите эту ячейку еще раз (Shift+Enter)")
else:
    zip_path = zip_files[0]
    print(f"✓ Найден ZIP: {zip_path}")
    print("Распаковка проекта...")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    
    print("✓ Проект распакован")
    
    # Ищем папку opencode
    opencode_path = None
    if os.path.exists('/content/opencode'):
        opencode_path = '/content/opencode'
    elif os.path.exists('/content/magic-artifact'):
        opencode_path = '/content/magic-artifact'
    else:
        # Ищем первую папку
        for item in os.listdir('/content'):
            if os.path.isdir(f'/content/{item}') and item != '__pycache__':
                opencode_path = f'/content/{item}'
                break
    
    if opencode_path:
        os.chdir(opencode_path)
        print(f"✓ Переходим в: {opencode_path}")
        print()
        print("Файлы проекта:")
        files = os.listdir('.')
        for f in sorted(files)[:15]:  # Показываем первые 15
            print(f"  • {f}")
        print()
        print("✓ ГОТОВО К СБОРКЕ")
    else:
        print("❌ ОШИБКА: Папка проекта не найдена!")

In [ ]:
# ============================================
# ЯЧЕЙКА 2: УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================

print("Обновление системы...")
!apt-get update > /dev/null 2>&1

print("Установка Java...")
!apt-get install -y openjdk-11-jdk-headless > /dev/null 2>&1

print("Установка Python инструментов...")
!pip install --upgrade pip setuptools wheel > /dev/null 2>&1
!pip install buildozer cython > /dev/null 2>&1

print()
print("✓ Java установлена")
!java -version 2>&1 | grep version
print()
print("✓ buildozer установлен")
!buildozer --version
print()
print("✓ Все зависимости установлены!")

In [ ]:
# ============================================
# ЯЧЕЙКА 3: УСТАНОВКА ANDROID SDK
# ============================================
# ВНИМАНИЕ: Это может занять 10-15 минут!

import os
import subprocess

print("⏳ Установка Android SDK...")
print("Это займет 10-15 минут, не закрывайте браузер!")
print()

sdk_path = os.path.expanduser("~/android-sdk")
os.makedirs(sdk_path, exist_ok=True)

os.chdir(sdk_path)

print("Скачивание Android SDK Command-line Tools...")
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
print("✓ Скачан")

print("Распаковка...")
!unzip -q commandlinetools-linux-*.zip
!rm commandlinetools-linux-*.zip
print("✓ Распакован")

print()
print("Установка компонентов SDK (платформа, build-tools, NDK)...")
!yes | ./cmdline-tools/bin/sdkmanager --install "platforms;android-31" "build-tools;31.0.0" "ndk;25.1.8937393" 2>/dev/null || true
print()
print("✓ Android SDK установлен!")
print()
print("Информация:")
print(f"  SDK путь: {sdk_path}")
print(f"  Android API: 31")
print(f"  NDK версия: 25.1.8937393")

In [ ]:
# ============================================
# ЯЧЕЙКА 4: СБОРКА APK (ГЛАВНАЯ!)
# ============================================
# ВНИМАНИЕ: Это может занять 10-20 минут!

import os
import subprocess
import glob

print("="*60)
print("🔨 НАЧАЛО СБОРКИ APK")
print("="*60)
print()
print("⏳ Это займет 10-20 минут...")
print("Будет много текста - это нормально!")
print()
print("-"*60)

# Устанавливаем переменные окружения
os.environ["ANDROID_SDK_ROOT"] = os.path.expanduser("~/android-sdk")
os.environ["ANDROID_NDK_ROOT"] = os.path.expanduser("~/android-sdk/ndk/25.1.8937393")

# ВАЖНО: Переходим в папку проекта
# Ищем папку с buildozer.spec
project_dir = None

# Проверяем возможные пути
possible_paths = [
    '/content/opencode',
    '/content/magic-artifact',
    '/workspace',
]

# Добавляем поиск в /content
for item in os.listdir('/content'):
    path = f'/content/{item}'
    if os.path.isdir(path) and os.path.exists(f'{path}/buildozer.spec'):
        possible_paths.insert(0, path)

# Ищем папку с buildozer.spec
for path in possible_paths:
    if os.path.exists(path) and os.path.exists(os.path.join(path, 'buildozer.spec')):
        project_dir = path
        break

if not project_dir:
    print("❌ ОШИБКА: buildozer.spec не найден!")
    print()
    print(f"Текущая папка: {os.getcwd()}")
    print("\nПопытка найти в стандартных местах...")
    for path in possible_paths:
        if os.path.exists(path):
            print(f"\n  📁 {path}")
            if os.path.exists(os.path.join(path, 'buildozer.spec')):
                print(f"     ✓ buildozer.spec найден!")
            else:
                print(f"     ❌ buildozer.spec не найден")
    print()
    print("Решение:")
    print("1. Убедитесь что ZIP был загружен правильно (ячейка 1)")
    print("2. Переустановите ZIP")
    print("3. Повторите ячейку 1")
else:
    print(f"✓ Папка проекта: {project_dir}")
    os.chdir(project_dir)
    print(f"✓ Переходим в: {os.getcwd()}")
    print(f"✓ buildozer.spec найден")
    print()
    
    # Сборка
    print("Запуск buildozer...")
    print()
    result = subprocess.run(["buildozer", "android", "debug"], 
                          capture_output=False,
                          text=True)
    
    print()
    print("-"*60)
    
    if result.returncode == 0:
        print()
        print("="*60)
        print("✅ APK УСПЕШНО СОБРАН!")
        print("="*60)
        print()
        print("Следующий шаг: запустите ячейку 5")
        print("(Там будет информация как скачать APK)")
    else:
        print()
        print("="*60)
        print("❌ ОШИБКА ПРИ СБОРКЕ")
        print("="*60)
        print()
        print("Проверьте вывод выше на наличие ошибок.")

In [ ]:
# ============================================
# ЯЧЕЙКА 5: ИНФОРМАЦИЯ О СКАЧИВАНИИ
# ============================================

import os
import glob

print("="*60)
print("📥 ИНФОРМАЦИЯ О СКАЧИВАНИИ APK")
print("="*60)
print()

# Ищем APK файлы
apk_files = glob.glob("bin/*.apk")

if apk_files:
    apk_path = apk_files[0]
    file_size = os.path.getsize(apk_path) / (1024*1024)
    
    print(f"✓ APK найден!")
    print()
    print(f"Файл:    {os.path.basename(apk_path)}")
    print(f"Размер:  {file_size:.1f} MB")
    print(f"Путь:    {apk_path}")
    print()
    print("-"*60)
    print()
    print("СКАЧИВАНИЕ:")
    print()
    print("1. Слева нажмите значок 'Files' (📁)")
    print()
    print("2. Откройте папку: bin")
    print()
    print("3. Найдите файл: magicartifact-0.1-debug.apk")
    print()
    print("4. Щелкните правой кнопкой → Download")
    print()
    print("-"*60)
    print()
    print("УСТАНОВКА НА ПЛАНШЕТ:")
    print()
    print("Подключите планшет по USB и выполните:")
    print()
    print("  adb install -r magicartifact-0.1-debug.apk")
    print()
    print("-"*60)
    print()
    print("✅ ВСЕ ГОТОВО!")
    
else:
    print("❌ APK файлы не найдены в bin/")
    print()
    print("Возможные причины:")
    print("1. Ячейка 4 (сборка) не была успешно выполнена")
    print("2. Произошла ошибка при сборке")
    print()
    print("Проверьте вывод ячейки 4 на наличие ошибок.")